# EvoNAS on Colab (A100)

Runs the evolutionary search with and without weight inheritance and compares them.
**Runtime > Change runtime type > A100 GPU** before starting.

Order: 1 setup, 2 smoke test (also measures the real time per candidate), 3 full runs, 4 compare.
Everything is written to Google Drive and every run **resumes automatically** if Colab disconnects:
just re-run cells 1 and 3 again.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/evonas_results'   # results + checkpoints survive disconnects here
!mkdir -p {OUT}

In [ ]:
import os
REPO = 'https://github.com/VedantPandhare/Evo-Neural-Architechtural-search.git'
if not os.path.exists('/content/evonas'):
    !git clone {REPO} /content/evonas
%cd /content/evonas
!git pull --ff-only
!pip install -q -r requirements.txt

In [ ]:
import torch
print(torch.__version__, '| cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > A100 GPU'
torch.backends.cuda.matmul.allow_tf32 = True   # A100 speed-up for dense layers (conv TF32 is already on)

In [ ]:
# quick sanity check of the code on this machine (CPU-only tests skipped: they need no GPU and are slow)
!python -m pytest -q tests/test_evolution.py tests/test_inheritance.py tests/test_run_logger.py tests/test_search.py tests/test_checkpoint.py

## 2. Smoke test (about a minute or two)
Tiny run to confirm everything works on the GPU, then it reports the **measured seconds per candidate**
and extrapolates the cost of the full runs. Use that number to decide the budget in `configs/search_*.yaml`.

In [ ]:
!python main.py --config configs/smoke.yaml --output-dir {OUT} --run-name smoke_colab

In [ ]:
import json, yaml

def read_candidates(run):
    with open(f'{OUT}/{run}/candidates.jsonl') as f:
        return [json.loads(l) for l in f]

recs = read_candidates('smoke_colab')
per_candidate = sum(r['train_seconds'] for r in recs) / len(recs)
cfg = yaml.safe_load(open('configs/search_inherit.yaml'))
e, t = cfg['evolution'], cfg['training']
print(f'smoke: {len(recs)} candidates, mean {per_candidate:.1f}s each '
      f'(smoke trains on {yaml.safe_load(open("configs/smoke.yaml"))["training"]["max_train_samples"]} images x '
      f'{yaml.safe_load(open("configs/smoke.yaml"))["training"]["epochs"]} epochs)')
scale = (t['max_train_samples'] * t['epochs']) / (2000 * 2)
n_evals = e['population_size'] + (e['generations'] - 1) * round(e['population_size'] * (1 - e['elite_fraction']))
est_h = per_candidate * scale * n_evals / 3600
print(f'full config: ~{n_evals} evaluations; rough estimate {est_h:.1f} h per run, {2 * est_h:.1f} h for both arms')
print('(rough: assumes training time scales with images x epochs; the first candidates include CUDA warm-up)')

## 3. Full runs
Run **inheritance ON** and **OFF** with the same seed and budget (the two configs differ only in that flag).
Each cell resumes automatically if a checkpoint exists. For the paper, repeat with several seeds
(change `SEED`; use at least 3).

In [ ]:
import os

SEED = 0

def run(config, run_name):
    resume = '--resume' if os.path.exists(f'{OUT}/{run_name}/checkpoint.pt') else ''
    !python main.py --config {config} --output-dir {OUT} --run-name {run_name} --seed {SEED} {resume}

In [ ]:
run('configs/search_inherit.yaml', f'search_inherit_seed{SEED}')

In [ ]:
run('configs/search_scratch.yaml', f'search_scratch_seed{SEED}')

## 4. Compare
Fitness per generation, cumulative wall-clock time, fraction of parameters inherited, and population diversity.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

runs = {'inheritance ON': f'search_inherit_seed{SEED}', 'inheritance OFF': f'search_scratch_seed{SEED}'}
gens = {k: pd.read_csv(f'{OUT}/{v}/generations.csv') for k, v in runs.items()}

fig, ax = plt.subplots(1, 4, figsize=(20, 4))
for name, g in gens.items():
    ax[0].plot(g.generation, g.best, label=name)
    ax[1].plot(g.generation, g.cumulative_seconds / 3600, label=name)
    ax[2].plot(g.generation, g.mean_inherit_fraction, label=name)
    ax[3].plot(g.generation, g.diversity, label=name)
for a, title in zip(ax, ['best fitness', 'cumulative hours (wall-clock)', 'mean inherited fraction', 'population diversity']):
    a.set_title(title); a.set_xlabel('generation'); a.legend()
plt.tight_layout(); plt.savefig(f'{OUT}/comparison_seed{SEED}.png', dpi=150); plt.show()

for name, v in runs.items():
    s = json.load(open(f'{OUT}/{v}/summary.json'))
    print(f"{name}: best fitness {s['best_fitness']:.4f} | total {s['total_gpu_hours_wallclock']:.2f} h")

In [ ]:
# accuracy vs parameter count of everything evaluated (the Pareto trade-off)
fig, ax = plt.subplots(figsize=(6, 4))
for name, v in runs.items():
    c = pd.DataFrame(read_candidates(v))
    ax.scatter(c.n_params, c.accuracy, s=14, alpha=.6, label=name)
ax.set_xscale('log'); ax.set_xlabel('parameters'); ax.set_ylabel('validation accuracy'); ax.legend()
plt.savefig(f'{OUT}/pareto_seed{SEED}.png', dpi=150); plt.show()

## 5. Final evaluation (do this once, after all searches are finished)
The searches only ever see the 5k validation split. The official 10k **test set** is used exactly once, here,
by retraining the best architecture from each arm on all 50k training images. Not implemented as a cell yet:
it needs the Stage 7 baseline/final-training script, which is the next piece of work.